# Dream Arcade on Kaggle

Turn on **GPU T4 x2** (or the single GPU your account offers). Confirm the weekly quota on your account page before a long run. Sessions end at 12 hours; this notebook checkpoints every 200 steps and is meant to be resumed.

Upload the repo as a Kaggle dataset, or clone it in the first cell. Do not commit Atari ROMs. Original Pong needs no ROM. Set `DATA=atari_pong` only after `import ale_py` works in this image.


In [ ]:
import os, subprocess, sys
from pathlib import Path
ROOT = Path('/kaggle/working/dream-arcade')
if not ROOT.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/namanraii/dream-arcade.git', str(ROOT)])
os.chdir(ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[metrics,track]'])
print('cuda', __import__('torch').cuda.is_available())


In [ ]:
# About 100k frames. For the 300k set, use --config collect_v1.
# Add expert_path=outputs/expert/expert.pt after train-expert if you want the 40% expert to be a learned policy.
!dreamarcade collect data=pong episodes=400 episode_length=256 out_dir=/kaggle/working/data-pong seed=0


In [ ]:
# Resume by setting resume=/kaggle/working/v2/latest.pt on the next session.
!dreamarcade train-v2 data_dir=/kaggle/working/data-pong out_dir=/kaggle/working/v2 device=cuda wandb=false save_every=200 steps=100000


In [ ]:
!dreamarcade train-v1 data_dir=/kaggle/working/data-pong out_dir=/kaggle/working/v1 device=cuda save_every=200
!dreamarcade train-reward data_dir=/kaggle/working/data-pong out_dir=/kaggle/working/reward device=cuda
!dreamarcade eval data_dir=/kaggle/working/data-pong checkpoint=/kaggle/working/v2/latest.pt reward_checkpoint=/kaggle/working/reward/latest.pt out_dir=/kaggle/working/eval device=cuda
